# ACOS — Training Pipeline
Splits the synthetic dataset into train/val sets and trains a YOLOv11m model for automatic checkout object detection.

## 1. Installations

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import shutil
import random
from pathlib import Path
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 3. Configuration
- `DATA_ROOT`: path to the synthetic dataset (from Notebook 1 output)
- `OUTPUT_ROOT`: where split data will be written
- `VAL_RATIO`: fraction of data held out for validation (default 20%)
- `SEED`: ensures reproducibility of the random split

In [3]:

DATA_ROOT   = Path("/kaggle/input/datasets/mayaralabidi/acos-synthetic-dataset/data")
OUTPUT_ROOT = Path("/kaggle/working/data_split")

IMAGES_DIR  = DATA_ROOT / "images"
LABELS_DIR  = DATA_ROOT / "labels"

VAL_RATIO   = 0.20   # 20% val → 400 images, 1600 train
SEED        = 42


## 4. Train/Val Split
Randomly shuffles all images and splits them according to `VAL_RATIO`. Copies images and their corresponding label files into separate `train/` and `val/` subdirectories. Images with no label file are treated as background samples.

In [4]:
def split_dataset():
    random.seed(SEED)

    # Collect all image paths
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    all_images = sorted([
        p for p in IMAGES_DIR.iterdir()
        if p.suffix.lower() in image_exts
    ])

    if not all_images:
        raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

    print(f"Total images found : {len(all_images)}")

    # Shuffle and split
    random.shuffle(all_images)
    n_val   = int(len(all_images) * VAL_RATIO)
    n_train = len(all_images) - n_val

    val_images   = all_images[:n_val]
    train_images = all_images[n_val:]

    print(f"Train              : {len(train_images)}")
    print(f"Val                : {len(val_images)}")

    # Create output directories
    for split in ("train", "val"):
        (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Copy files
    missing_labels = []

    def copy_split(image_list, split_name):
        for img_path in image_list:
            label_path = LABELS_DIR / (img_path.stem + ".txt")

            # Copy image
            shutil.copy(img_path,
                        OUTPUT_ROOT / "images" / split_name / img_path.name)

            # Copy label if it exists
            if label_path.exists():
                shutil.copy(label_path,
                            OUTPUT_ROOT / "labels" / split_name / label_path.name)
            else:
                missing_labels.append(img_path.name)

    copy_split(train_images, "train")
    copy_split(val_images,   "val")

    if missing_labels:
        print(f"\n⚠️  {len(missing_labels)} images had no label file "
              f"(treated as background):")
        for name in missing_labels[:10]:
            print(f"   {name}")
        if len(missing_labels) > 10:
            print(f"   ... and {len(missing_labels) - 10} more")

    print("\n✅ Split complete.")
    print(f"   Output: {OUTPUT_ROOT}")

    # Verify counts
    for split in ("train", "val"):
        n_img = len(list((OUTPUT_ROOT / "images" / split).iterdir()))
        n_lbl = len(list((OUTPUT_ROOT / "labels" / split).iterdir()))
        print(f"   {split:5s} → {n_img} images, {n_lbl} labels")

    return train_images, val_images

## 5. Write `data.yaml`
Generates the YOLO configuration file with the correct paths and all 45 class names for the split dataset.

In [5]:

def write_yaml():
    yaml_content = f"""path: {OUTPUT_ROOT}
train: images/train
val:   images/val

nc: 45
names: [
  'choco_esprit_de_fete', 'milk_delice', 'juice_diva',
  'soapbar_dove_shea', 'butter_jadida', 'vanillinatedsugar',
  'disinfectant_cnett', 'cocoapowder', 'pril',
  'conditioner_avilea', 'showergel', 'riso_scotti',
  'flour', 'yogurt_danette', 'choc',
  'pasta_spaghetti', 'rice', 'bathfoam_malizia',
  'soapbar_dove_lavender', 'dryyeast_smartchef',
  'butter_delice', 'judy', 'carolin',
  'vinegar', 'dryyeast_lapatissiere', 'choco_coating',
  'toothpaste_colgate', 'shampoo_elseve_hya', 'bakingpowder',
  'pasta_fell', 'teabags_camomilia', 'chocoline',
  'pasta_canelloni', 'yogurt_delice', 'coffee_bondin',
  'chantilly_vanoise', 'shampoo_elseve_gly', 'shampoo_elvive',
  'orzo', 'soap_lilas', 'mustard',
  'harissa', 'liquidsoap_naya', 'chocospread_vanoise', 'canned_peas'
]
"""
    yaml_path = OUTPUT_ROOT / "data.yaml"
    yaml_path.write_text(yaml_content)
    print(f"\n✅ data.yaml written to {yaml_path}")
    return yaml_path

## 6. Class Distribution Check
Verifies that all 45 product classes appear in **both** splits. Flags any class with zero or very few validation samples — important for rare classes like `harissa` which have limited instances in the dataset.

In [6]:
def check_class_distribution(train_images, val_images):
    """
    Verify that all 45 classes appear in both splits.
    Catches the edge case where a rare class (e.g. harissa with only
    66 instances) ends up entirely in one split by bad luck.
    """
    from collections import Counter

    def count_classes(image_list):
        counts = Counter()
        for img_path in image_list:
            label_path = LABELS_DIR / (img_path.stem + ".txt")
            if label_path.exists():
                for line in label_path.read_text().strip().splitlines():
                    if line.strip():
                        cls = int(line.split()[0])
                        counts[cls] += 1
        return counts

    print("\n  Checking class distribution...")
    train_counts = count_classes(train_images)
    val_counts   = count_classes(val_images)

    print(f"  {'Class':35s}  {'Train':>6}  {'Val':>5}  {'Val%':>5}")
    print("  " + "-" * 58)

    warned = False
    for cls in range(45):
        t = train_counts.get(cls, 0)
        v = val_counts.get(cls, 0)
        pct = 100 * v / (t + v) if (t + v) > 0 else 0
        flag = ""
        if v == 0:
            flag = "  ⚠️  NO VAL SAMPLES"
            warned = True
        elif pct < 10:
            flag = "  ⚠️  LOW VAL"
            warned = True
        print(f"  {cls:3d}  {'':31s}  {t:6d}  {v:5d}  {pct:4.0f}%{flag}")

    if not warned:
        print("\n  ✅ All classes present in both splits.")
    else:
        print("\n  ⚠️  Some classes are under-represented in val.")
        print("     Consider stratified split (see below).")

## 7. Run Pipeline
Executes the full pipeline in order:
1. Split dataset
2. Check class distribution
3. Write `data.yaml`

In [7]:

if __name__ == "__main__":
    # Step 1: basic random split
    train_imgs, val_imgs = split_dataset()

    # Step 2: verify class distribution
    check_class_distribution(train_imgs, val_imgs)

    # Step 3: write yaml
    yaml_path = write_yaml()

Total images found : 2000
Train              : 1600
Val                : 400

✅ Split complete.
   Output: /kaggle/working/data_split
   train → 1600 images, 1600 labels
   val   → 400 images, 400 labels

  Checking class distribution...
  Class                                 Train    Val   Val%
  ----------------------------------------------------------
    0                                      166     45    21%
    1                                      230     61    21%
    2                                      247     69    22%
    3                                      157     32    17%
    4                                      171     39    19%
    5                                      364     86    19%
    6                                      136     36    21%
    7                                      123     31    20%
    8                                      106     22    17%
    9                                       67     17    20%
   10                          

## 8. Train YOLOv11m
Fine-tunes a pretrained YOLOv11m model on the split dataset.

| Parameter | Value | Notes |
|---|---|---|
| `epochs` | 45 | Extended from 20 — loss was still improving |
| `imgsz` | 640 | Standard YOLO input size |
| `batch` | 16 | |
| `patience` | 15 | Early stopping if val mAP plateaus |
| `save_period` | 10 | Checkpoint every 10 epochs |

In [8]:
model = YOLO("yolo11m.pt")

model.train(
    data="/kaggle/working/data_split/data.yaml",
    epochs=45,          # was 20 — loss was still improving
    imgsz=640,
    batch=16,
    patience=15,        # stop early if val mAP stops improving
    save_period=10,     # checkpoint every 10 epochs
)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=45, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x788a903139b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,